# VirtualFit — Fine-tuning IDM-VTON on a Custom Try-On Dataset

**Goal.** Take the pretrained **IDM-VTON** virtual try-on model and *adapt it to our
own data* — 10,000 pairs of `(garment flat image, person wearing that garment)` for
shirts and pants — so the renders look better on *our* domain (our garments, lighting,
body types).

---
## 1. What IDM-VTON is (architecture you must be able to explain)

IDM-VTON ("Improving Diffusion Models for Authentic Virtual Try-on in the Wild") is a
**latent diffusion** model built on **SDXL inpainting**. It has these parts:

```
            garment image ──► GarmentNet (UNet_Encoder)  ─┐  reference features
                                                          │  (garment appearance)
 person (masked) ─VAE─► latent ─┐                         ▼
 agnostic MASK ────────────────┤──► TryonNet (main UNet) ──► predicted noise ε̂
 DensePose pose ─VAE─► latent ──┤        ▲          ▲
 garment image ─► CLIP-image ───┘  IP-Adapter   text (CLIP) "model is wearing ..."
```

- **TryonNet** (`src/unet_hacked_tryon.py`) — the main SDXL-inpaint UNet we denoise with.
  Its input is the noisy person-latent **concatenated** with the mask, the masked-person
  latent, and the DensePose latent (micro-conditioning for SDXL on top).
- **GarmentNet** (`src/unet_hacked_garmnet.py`, loaded as `unet_encoder`) — a parallel UNet
  that encodes the *flat garment* and exposes intermediate **reference features** that are
  injected into TryonNet's self-attention so the garment's texture/pattern is preserved.
- **IP-Adapter image encoder** (CLIP vision) — a second, *global* garment cue.
- **VAE** (frozen) — image ⇄ latent. **Text encoders** (frozen) — encode the prompt.

## 2. What we actually fine-tune (and why)

We fine-tune **TryonNet with LoRA** (Low-Rank Adaptation) and keep VAE, text encoders,
the CLIP image encoder, and (by default) GarmentNet **frozen**.

**Why LoRA and not full fine-tuning?**
- Full fine-tuning of *two* SDXL UNets at 768×1024 needs **~40–60 GB VRAM** + huge optimizer
  state — infeasible on a single L4 (24 GB) or A100-40GB.
- LoRA adds tiny rank-`r` matrices to the attention projections: we train **<1%** of the
  weights, so it fits in memory, trains far faster, is **less prone to overfitting** on a
  10k set, and the result is a small (~10–100 MB) adapter you can ship next to the base model.
- This is the standard, defensible choice for adapting large diffusion models on limited data.

## 3. The training objective (the "loss")

Standard **DDPM denoising loss** (ε-prediction): we add known Gaussian noise `ε` to the
person latent at a random timestep `t`, ask TryonNet to predict that noise given all the
conditioning, and minimize

```
L = E_{x0, ε, t} [ || ε − ε̂_θ(z_t, t, conditioning) ||² ]   (optionally Min-SNR-γ weighted)
```

Minimizing this teaches the model to reconstruct *our* people-in-garments from noise.

> **Read the cells top-to-bottom.** Comments explain every step, and the final two
> sections are (a) **evaluation metrics** and (b) a **Professor Q&A** with model answers.


In [ ]:
# 1. Confirm GPU + report VRAM. Fine-tuning needs an L4 (24 GB) at minimum; A100 ideal.
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU (L4/A100)."
name = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f"Device: {name} | VRAM: {vram} GB")
# LoRA @ 768x1024 fits ~24 GB with bf16 + grad-checkpointing + 8-bit Adam.
# If <24 GB, drop RES to 512x384 (see CONFIG) or reduce LoRA rank / batch size.

In [ ]:
# 2. Clone IDM-VTON and install the EXACT training stack.
#    Same pins as the inference server (torch 2.2.2 / diffusers 0.25.0 / numpy<2 for
#    detectron2) PLUS the fine-tuning tools: peft (LoRA), bitsandbytes (8-bit Adam),
#    accelerate (mixed precision + device placement), wandb (logging), torchmetrics +
#    lpips + clean-fid (evaluation).
#    >>> AFTER this cell: Runtime > Restart session, then run from cell 3. <<<
import os
os.environ["FORCE_CUDA"] = "1"
%cd /content
![ -d IDM-VTON ] || git clone https://github.com/yisol/IDM-VTON.git
!pip -q install torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -2
!pip -q install "numpy==1.26.4" 2>&1 | tail -1
!pip -q install "git+https://github.com/facebookresearch/detectron2.git" 2>&1 | tail -2
!pip -q install "diffusers==0.25.0" "transformers==4.36.2" "accelerate==0.25.0" "peft==0.7.1" "huggingface_hub==0.20.3" onnxruntime "scipy==1.11.4" "einops==0.7.0" "torchmetrics==1.2.1" opencv-python "fvcore==0.1.5.post20221221" cloudpickle "omegaconf==2.3.0" pycocotools av 2>&1 | tail -3
# Fine-tuning extras:
!pip -q install "bitsandbytes==0.43.1" wandb lpips clean-fid 2>&1 | tail -3
print("INSTALL DONE -> Runtime > Restart session, then run cells 3+ (do NOT re-run this cell).")

## 4. Dataset layout (organise your 10k images like this)

IDM-VTON is trained on **paired** data (VITON-HD format). Put your raw images here on
Drive/Colab and the preprocessing cell will generate the rest:

```
/content/dataset/
  train/
    image/      0001.jpg ...   # the PERSON wearing the garment   (ground truth target)
    cloth/      0001.jpg ...   # the FLAT garment (shirt or pant)  (same stem as image/)
  test/                        # a held-out split for validation (same structure)
    image/ ...
    cloth/ ...
  category.txt                 # optional: "<stem> <upper|lower>" per line; default=upper
```

Rules:
- **Filenames pair by stem**: `image/0001.jpg` ↔ `cloth/0001.jpg` are the same outfit.
- Keep aspect ratio ~3:4 (portrait). We resize to 768×1024 (or 512×384 if low VRAM).
- **Split before training** (e.g. 9k train / 1k test). *Never* let a test pair leak into
  train — that inflates metrics and your professor *will* ask about it.

The preprocessing cell computes, per person image, the two conditioning maps IDM-VTON
needs and caches them so each epoch is fast:
- **agnostic mask** — region to be re-painted (the garment area), from human-parsing + OpenPose.
- **DensePose** — dense body-surface map that anchors how the garment should drape.


In [ ]:
# 4. CONFIG — every hyperparameter in one place (log this to W&B for reproducibility).
from dataclasses import dataclass, asdict, field

@dataclass
class CFG:
    # --- data ---
    data_root: str = "/content/dataset"
    cache_dir: str = "/content/dataset_cache"   # precomputed mask/densepose go here
    width: int  = 768                            # 512 if VRAM-limited (also set height 512->? keep 4:3)
    height: int = 1024                           # use 384x512 pair on small GPUs
    default_category: str = "upper_body"         # upper_body | lower_body | dresses

    # --- model ---
    base_repo: str = "yisol/IDM-VTON"            # pretrained weights we adapt
    train_garmentnet: bool = False               # keep GarmentNet frozen (memory); True = also LoRA it

    # --- LoRA ---
    lora_rank: int = 32                          # capacity of the adapter (8/16/32/64). Higher=more capacity+VRAM
    lora_alpha: int = 32                         # scaling; commonly == rank
    lora_dropout: float = 0.0

    # --- optimisation ---
    lr: float = 1e-4                             # LoRA tolerates higher LR than full FT
    lr_warmup_steps: int = 200
    max_train_steps: int = 6000                  # ~ (9000 imgs / (bs*accum)) * epochs. Tune.
    train_batch_size: int = 1                    # per-device; 768x1024 SDXL is heavy
    grad_accum: int = 4                          # effective batch = bs * grad_accum = 4
    gradient_checkpointing: bool = True          # trade compute for memory (needed on L4)
    mixed_precision: str = "bf16"                # bf16 on A100/L4; "fp16" if bf16 unsupported
    max_grad_norm: float = 1.0
    seed: int = 42
    snr_gamma: float = 5.0                       # Min-SNR-gamma loss weighting (None to disable)

    # --- logging / checkpoints / validation ---
    project: str = "virtualfit-idmvton-finetune"
    run_name: str = "lora-r32-bf16"
    output_dir: str = "/content/idmvton_lora_out"
    log_every: int = 10                          # steps between loss logs
    val_every: int = 500                         # steps between validation renders
    save_every: int = 1000                       # steps between checkpoints
    num_val_samples: int = 4

cfg = CFG()
os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.cache_dir, exist_ok=True)
print(asdict(cfg))

In [ ]:
# 5. Load the THREE preprocessing models (same ones the inference server uses):
#    human-parsing (SCHP/ATR), OpenPose, and DensePose (detectron2 apply_net).
#    These produce the agnostic mask + dense body map for every training image.
import os, sys, torch
os.chdir("/content/IDM-VTON")
sys.path.insert(0, "/content/IDM-VTON")
sys.path.insert(0, "/content/IDM-VTON/gradio_demo")   # apply_net / densepose / utils_mask live here

from preprocess.humanparsing.run_parsing import Parsing
from preprocess.openpose.run_openpose import OpenPose
parsing_model = Parsing(0)
openpose_model = OpenPose(0)
openpose_model.preprocessor.body_estimation.model.to("cuda")

# detectron2 has a missing symbol in this combo; mock it (same patch as the server).
import detectron2.structures.boxes as _boxes
if not hasattr(_boxes, "matched_boxlist_iou"):
    _boxes.matched_boxlist_iou = getattr(_boxes, "pairwise_iou", lambda *a, **k: None)
import apply_net
from detectron2.data.detection_utils import convert_PIL_to_numpy, _apply_exif_orientation

import importlib.util
_spec = importlib.util.spec_from_file_location("utils_mask", "/content/IDM-VTON/gradio_demo/utils_mask.py")
utils_mask = importlib.util.module_from_spec(_spec); sys.modules["utils_mask"] = utils_mask
_spec.loader.exec_module(utils_mask)
get_mask_location = utils_mask.get_mask_location

# DensePose arg parser (relative ./configs path -> needs cwd == /content/IDM-VTON, set above).
_dp_args = apply_net.create_argument_parser().parse_args((
    "show", "./configs/densepose_rcnn_R_50_FPN_s1x.yaml",
    "./ckpt/densepose/model_final_162be9.pkl", "dp_segm", "-v",
    "--opts", "MODEL.DEVICE", "cuda"))
# NOTE: ./ckpt symlinks are created by the inference notebook's cell 4. If missing, run that
# cell once (or snapshot_download the repo and symlink ckpt/{densepose,humanparsing,openpose}).
print("preprocess models ready")

In [ ]:
# 6. Precompute + cache (agnostic MASK, DensePose) for every person image.
#    Doing this ONCE up front means the DataLoader just reads .png/.npy each epoch
#    (DensePose at train time would be far too slow). Re-runnable: it skips done files.
import glob, numpy as np
from PIL import Image
from tqdm.auto import tqdm

CAT = {"upper": "upper_body", "lower": "lower_body", "upper_body": "upper_body",
       "lower_body": "lower_body", "dresses": "dresses", "full": "dresses"}

def load_categories(split):
    # optional category.txt: "<stem> <upper|lower>"; default to cfg.default_category
    path = os.path.join(cfg.data_root, "category.txt")
    m = {}
    if os.path.exists(path):
        for line in open(path):
            parts = line.split()
            if len(parts) >= 2: m[parts[0]] = CAT.get(parts[1], cfg.default_category)
    return m

def make_mask_and_pose(person_pil, category):
    # 1) agnostic mask from OpenPose keypoints + human parsing (the region to repaint)
    kpts = openpose_model(person_pil.resize((384, 512)))
    parse, _ = parsing_model(person_pil.resize((384, 512)))
    mask, _ = get_mask_location("hd", category, parse, kpts)
    mask = mask.resize((cfg.width, cfg.height))
    # 2) DensePose body map (anchors garment draping)
    arg = _apply_exif_orientation(person_pil.resize((384, 512)))
    arg = convert_PIL_to_numpy(arg, format="BGR")
    pose = _dp_args.func(_dp_args, arg)[:, :, ::-1]
    pose = Image.fromarray(pose).resize((cfg.width, cfg.height))
    return mask.convert("L"), pose.convert("RGB")

def preprocess_split(split):
    cats = load_categories(split)
    img_dir = os.path.join(cfg.data_root, split, "image")
    out_mask = os.path.join(cfg.cache_dir, split, "mask"); os.makedirs(out_mask, exist_ok=True)
    out_pose = os.path.join(cfg.cache_dir, split, "densepose"); os.makedirs(out_pose, exist_ok=True)
    files = sorted(glob.glob(os.path.join(img_dir, "*")))
    print(split, "->", len(files), "images")
    for f in tqdm(files):
        stem = os.path.splitext(os.path.basename(f))[0]
        mp = os.path.join(out_mask, stem + ".png"); pp = os.path.join(out_pose, stem + ".png")
        if os.path.exists(mp) and os.path.exists(pp):  # resume-friendly
            continue
        person = Image.open(f).convert("RGB")
        cat = cats.get(stem, cfg.default_category)
        try:
            mask, pose = make_mask_and_pose(person, cat)
            mask.save(mp); pose.save(pp)
        except Exception as e:
            print("skip", stem, e)

for split in ("train", "test"):
    if os.path.isdir(os.path.join(cfg.data_root, split, "image")):
        preprocess_split(split)
print("preprocessing cached at", cfg.cache_dir)

In [ ]:
# 7. Dataset + DataLoader. Returns the tensors the training step needs, all in [-1,1].
import numpy as np, glob, random
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

to_tensor_norm = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])  # ->[-1,1]
mask_to_tensor = transforms.ToTensor()  # mask stays [0,1]

class VTONDataset(Dataset):
    def __init__(self, split):
        self.split = split
        self.img_dir   = os.path.join(cfg.data_root, split, "image")
        self.cloth_dir = os.path.join(cfg.data_root, split, "cloth")
        self.mask_dir  = os.path.join(cfg.cache_dir, split, "mask")
        self.pose_dir  = os.path.join(cfg.cache_dir, split, "densepose")
        self.stems = [os.path.splitext(os.path.basename(f))[0]
                      for f in sorted(glob.glob(os.path.join(self.img_dir, "*")))
                      if os.path.exists(os.path.join(self.mask_dir, os.path.splitext(os.path.basename(f))[0] + ".png"))]
        self.size = (cfg.width, cfg.height)
        print(split, "usable pairs:", len(self.stems))

    def __len__(self): return len(self.stems)

    def _load(self, d, stem, ext=(".jpg", ".png", ".jpeg")):
        for e in ext:
            p = os.path.join(d, stem + e)
            if os.path.exists(p): return Image.open(p).convert("RGB")
        raise FileNotFoundError(stem + " in " + d)

    def __getitem__(self, i):
        stem = self.stems[i]
        person = self._load(self.img_dir, stem).resize(self.size)
        cloth  = self._load(self.cloth_dir, stem).resize(self.size)
        pose   = Image.open(os.path.join(self.pose_dir, stem + ".png")).convert("RGB").resize(self.size)
        mask   = Image.open(os.path.join(self.mask_dir, stem + ".png")).convert("L").resize(self.size)
        return {
            "person": to_tensor_norm(person),     # target image (what we reconstruct)  [-1,1]
            "cloth":  to_tensor_norm(cloth),      # flat garment (GarmentNet + IP-adapter) [-1,1]
            "pose":   to_tensor_norm(pose),       # densepose conditioning               [-1,1]
            "mask":   mask_to_tensor(mask),       # 1 = repaint (garment area)           [0,1]
            "caption": "model is wearing a garment",  # replace with real captions if you have them
            "stem": stem,
        }

train_ds = VTONDataset("train")
train_dl = DataLoader(train_ds, batch_size=cfg.train_batch_size, shuffle=True,
                      num_workers=2, drop_last=True, pin_memory=True)

# Sanity-check one sample visually (great to show in your report).
import matplotlib.pyplot as plt
b = train_ds[0]
fig, ax = plt.subplots(1, 4, figsize=(12, 5))
for a, k in zip(ax, ["person", "cloth", "pose", "mask"]):
    t = b[k]
    img = (t.permute(1, 2, 0).numpy() * 0.5 + 0.5) if k != "mask" else t[0].numpy()
    a.imshow(img, cmap=None if k != "mask" else "gray"); a.set_title(k); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 8. Load the IDM-VTON model components we train with / condition on.
from huggingface_hub import snapshot_download
from diffusers import DDPMScheduler, AutoencoderKL
from transformers import (CLIPTextModel, CLIPTextModelWithProjection, AutoTokenizer,
                          CLIPVisionModelWithProjection, CLIPImageProcessor)
from src.unet_hacked_tryon import UNet2DConditionModel            # TryonNet (we LoRA this)
from src.unet_hacked_garmnet import UNet2DConditionModel as UNetGarm  # GarmentNet (frozen)

BASE = snapshot_download(repo_id=cfg.base_repo)
dtype = torch.bfloat16 if cfg.mixed_precision == "bf16" else torch.float16
device = "cuda"

noise_scheduler = DDPMScheduler.from_pretrained(BASE, subfolder="scheduler")  # TRAINING scheduler (DDPM)
vae            = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=torch.float32)  # keep fp32 for stable latents
unet           = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=dtype)   # TryonNet
garmentnet     = UNetGarm.from_pretrained(BASE, subfolder="unet_encoder", torch_dtype=dtype)        # GarmentNet
image_encoder  = CLIPVisionModelWithProjection.from_pretrained(BASE, subfolder="image_encoder", torch_dtype=dtype)
text_encoder_1 = CLIPTextModel.from_pretrained(BASE, subfolder="text_encoder", torch_dtype=dtype)
text_encoder_2 = CLIPTextModelWithProjection.from_pretrained(BASE, subfolder="text_encoder_2", torch_dtype=dtype)
tokenizer_1    = AutoTokenizer.from_pretrained(BASE, subfolder="tokenizer", use_fast=False)
tokenizer_2    = AutoTokenizer.from_pretrained(BASE, subfolder="tokenizer_2", use_fast=False)
feature_extractor = CLIPImageProcessor()

# Freeze everything that is NOT being LoRA-trained.
for m in (vae, garmentnet, image_encoder, text_encoder_1, text_encoder_2):
    m.requires_grad_(False); m.eval()
unet.requires_grad_(False)  # base UNet frozen; LoRA adds the only trainable params (next cell)

vae.to(device); image_encoder.to(device); text_encoder_1.to(device); text_encoder_2.to(device)
garmentnet.to(device); unet.to(device)

# >>> VERIFY conditioning shapes against THIS checkpoint (do not skip — exam-relevant) <<<
import inspect
print("TryonNet conv_in.in_channels =", unet.conv_in.in_channels,
      " (expect 13 = 4 noise + 4 pose + 1 mask + 4 masked-image)")
print("VAE scaling_factor =", vae.config.scaling_factor)
print("TryonNet.forward params:", list(inspect.signature(unet.forward).parameters)[:14], "...")

In [ ]:
# 9. Attach LoRA to TryonNet's attention layers and build the optimizer.
from peft import LoraConfig
from diffusers.optimization import get_scheduler

lora = LoraConfig(
    r=cfg.lora_rank, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
    init_lora_weights="gaussian",
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],  # attention projections
)
unet.add_adapter(lora)
if cfg.gradient_checkpointing:
    unet.enable_gradient_checkpointing()

# Only the LoRA params require grad -> tiny trainable set.
trainable = [p for p in unet.parameters() if p.requires_grad]
n_train = sum(p.numel() for p in trainable)
n_total = sum(p.numel() for p in unet.parameters())
print(f"Trainable (LoRA): {n_train/1e6:.2f} M  /  Total UNet: {n_total/1e6:.1f} M  "
      f"({100*n_train/n_total:.3f}%)")

# 8-bit Adam keeps optimizer state small (key for fitting on one GPU).
import bitsandbytes as bnb
optimizer = bnb.optim.AdamW8bit(trainable, lr=cfg.lr, betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8)
lr_sched = get_scheduler("cosine", optimizer=optimizer,
                         num_warmup_steps=cfg.lr_warmup_steps,
                         num_training_steps=cfg.max_train_steps)

In [ ]:
# 10. Accelerator (mixed precision + grad accumulation) + Weights & Biases.
from accelerate import Accelerator
from accelerate.utils import set_seed
import wandb

set_seed(cfg.seed)
accelerator = Accelerator(mixed_precision=cfg.mixed_precision,
                          gradient_accumulation_steps=cfg.grad_accum,
                          log_with="wandb")
# Paste your key when prompted, or run `wandb login` / set WANDB_API_KEY beforehand.
accelerator.init_trackers(cfg.project, config=asdict(cfg),
                          init_kwargs={"wandb": {"name": cfg.run_name}})

unet, optimizer, train_dl, lr_sched = accelerator.prepare(unet, optimizer, train_dl, lr_sched)
print("accelerator ready on", accelerator.device, "| mixed_precision =", cfg.mixed_precision)

In [ ]:
# 11. The training forward pass = ONE denoising step (this is the heart of the notebook).
#     It mirrors what TryonPipeline.__call__ does internally, but for a single random
#     timestep so we can compute a loss. Read every comment.
import torch.nn.functional as F

@torch.no_grad()
def encode_text(captions):
    # SDXL uses TWO text encoders; we concat their token embeds and take the 2nd's pooled.
    embeds, pooled = [], None
    for tok, te in ((tokenizer_1, text_encoder_1), (tokenizer_2, text_encoder_2)):
        ids = tok(captions, padding="max_length", max_length=tok.model_max_length,
                  truncation=True, return_tensors="pt").input_ids.to(device)
        out = te(ids, output_hidden_states=True)
        if hasattr(out, "text_embeds"): pooled = out.text_embeds        # pooled from encoder-2
        embeds.append(out.hidden_states[-2])                            # penultimate layer (SDXL convention)
    return torch.cat(embeds, dim=-1), pooled                            # (B,77,2048), (B,1280)

@torch.no_grad()
def vae_encode(x):  # x in [-1,1] -> scaled latent
    return vae.encode(x.to(vae.dtype)).latent_dist.sample() * vae.config.scaling_factor

def snr_weights(timesteps):
    # Min-SNR-gamma: down-weight easy (low-noise) steps so training is balanced.
    ac = noise_scheduler.alphas_cumprod.to(device)[timesteps]
    snr = ac / (1 - ac)
    if cfg.snr_gamma is None: return torch.ones_like(snr)
    return torch.stack([snr, cfg.snr_gamma * torch.ones_like(snr)], 1).min(1)[0] / snr

def training_step(batch):
    person = batch["person"].to(device); cloth = batch["cloth"].to(device)
    pose   = batch["pose"].to(device);   mask  = batch["mask"].to(device)
    bsz = person.shape[0]

    # --- targets & conditioning latents (VAE frozen) ---
    latents        = vae_encode(person)                              # what we add noise to
    masked_person  = person * (mask < 0.5)                           # garment area blanked
    masked_latents = vae_encode(masked_person)
    pose_latents   = vae_encode(pose)
    mask_lat = F.interpolate(mask, size=latents.shape[-2:])          # mask down to latent res

    # --- diffusion noising ---
    noise = torch.randn_like(latents)
    t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
    noisy = noise_scheduler.add_noise(latents, noise, t)

    # --- text + SDXL micro-conditioning (added_cond_kwargs) ---
    prompt_embeds, pooled = encode_text(list(batch["caption"]))
    cloth_embeds, _ = encode_text(["a photo of a garment"] * bsz)    # caption for GarmentNet branch
    add_time_ids = torch.tensor([cfg.height, cfg.width, 0, 0, cfg.height, cfg.width],
                                device=device, dtype=prompt_embeds.dtype).repeat(bsz, 1)
    added = {"text_embeds": pooled, "time_ids": add_time_ids}

    # --- GarmentNet reference features (garment appearance injected into TryonNet) ---
    cloth_latents = vae_encode(cloth)
    # GarmentNet returns (sample, reference_features). The 2nd is consumed by TryonNet.
    # VERIFY kwarg/return against src/unet_hacked_garmnet.py + tryon (see cell 8 prints).
    _, garment_features = garmentnet(cloth_latents, t, cloth_embeds.to(dtype),
                                     return_dict=False)

    # --- assemble TryonNet input (channel ORDER must match conv_in; verified in cell 8) ---
    # 13ch = noisy(4) + pose(4) + mask(1) + masked(4)
    unet_in = torch.cat([noisy, pose_latents, mask_lat, masked_latents], dim=1).to(dtype)

    model_pred = unet(unet_in, t, encoder_hidden_states=prompt_embeds.to(dtype),
                      added_cond_kwargs=added,
                      garment_features=garment_features,    # <-- verify this kwarg name
                      return_dict=False)[0]

    # --- loss: predicted noise vs true noise, Min-SNR weighted ---
    target = noise   # epsilon-prediction (SDXL default; if scheduler is v-pred, switch target)
    w = snr_weights(t)
    loss = (w * F.mse_loss(model_pred.float(), target.float(), reduction="none").mean([1, 2, 3])).mean()
    return loss

In [ ]:
# 12. Training loop — grad accumulation, clipping, W&B logging, periodic validation+ckpt.
import math, time
from tqdm.auto import tqdm

history = {"step": [], "loss": [], "lr": []}        # for the offline charts in cell 13
global_step = 0
unet.train()
pbar = tqdm(total=cfg.max_train_steps, desc="train")
done = False
while not done:
    for batch in train_dl:
        with accelerator.accumulate(unet):
            loss = training_step(batch)
            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_([p for p in unet.parameters() if p.requires_grad],
                                            cfg.max_grad_norm)
            optimizer.step(); lr_sched.step(); optimizer.zero_grad()

        if accelerator.sync_gradients:          # one real optimisation step happened
            global_step += 1; pbar.update(1)
            if global_step % cfg.log_every == 0:
                l = accelerator.gather(loss.detach()).mean().item()
                lr = lr_sched.get_last_lr()[0]
                history["step"].append(global_step); history["loss"].append(l); history["lr"].append(lr)
                accelerator.log({"train/loss": l, "train/lr": lr}, step=global_step)
                pbar.set_postfix(loss=f"{l:.4f}", lr=f"{lr:.2e}")
            if global_step % cfg.save_every == 0 and accelerator.is_main_process:
                p = os.path.join(cfg.output_dir, f"lora-step{global_step}")
                accelerator.unwrap_model(unet).save_pretrained(p)   # saves LoRA adapter only
                print("saved", p)
            if global_step % cfg.val_every == 0 and accelerator.is_main_process:
                try:
                    run_validation(global_step)   # defined in cell 14
                except Exception as e:
                    print("validation skipped:", e)
            if global_step >= cfg.max_train_steps:
                done = True; break
pbar.close()
# final save
final = os.path.join(cfg.output_dir, "lora-final")
accelerator.unwrap_model(unet).save_pretrained(final)
print("DONE. final LoRA ->", final)

In [ ]:
# 13. Loss + LR curves (also live in W&B, but embed these PNGs in your report).
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(history["step"], history["loss"]); ax[0].set_title("Training loss (MSE, Min-SNR)")
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].grid(alpha=.3)
# moving average to see the trend through the noise
import numpy as np
if len(history["loss"]) > 20:
    k = 20; ma = np.convolve(history["loss"], np.ones(k)/k, mode="valid")
    ax[0].plot(history["step"][k-1:], ma, lw=2, label="MA(20)"); ax[0].legend()
ax[1].plot(history["step"], history["lr"]); ax[1].set_title("Learning rate (cosine + warmup)")
ax[1].set_xlabel("step"); ax[1].set_ylabel("lr"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(cfg.output_dir, "curves.png"), dpi=120); plt.show()

In [ ]:
# 14. Validation render with the CURRENT model (and reusable after training).
#     Builds the FULL TryonPipeline so we get a real sampled image (not a one-step pred),
#     loads the trained LoRA into it, and renders a few held-out pairs for W&B + the report.
from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
import wandb

_val_pipe = None
def _build_pipe():
    global _val_pipe
    if _val_pipe is not None: return _val_pipe
    p = TryonPipeline.from_pretrained(
        BASE, unet=accelerator.unwrap_model(unet), vae=vae, feature_extractor=feature_extractor,
        text_encoder=text_encoder_1, text_encoder_2=text_encoder_2,
        tokenizer=tokenizer_1, tokenizer_2=tokenizer_2,
        scheduler=noise_scheduler, image_encoder=image_encoder, torch_dtype=dtype)
    p.unet_encoder = garmentnet
    p.to(device); _val_pipe = p; return p

@torch.no_grad()
def run_validation(step):
    pipe = _build_pipe(); pipe.unet.eval()
    test_ds = VTONDataset("test")
    idxs = list(range(min(cfg.num_val_samples, len(test_ds))))
    panels = []
    for i in idxs:
        s = test_ds[i]
        person = (s["person"].permute(1,2,0).numpy()*0.5+0.5)
        cloth_pil = Image.fromarray(((s["cloth"].permute(1,2,0).numpy()*0.5+0.5)*255).astype("uint8"))
        person_pil = Image.fromarray((person*255).astype("uint8"))
        mask_pil = Image.fromarray((s["mask"][0].numpy()*255).astype("uint8"))
        pose_t = s["pose"].unsqueeze(0).to(device, dtype)
        pe, npe, ppe, nppe = pipe.encode_prompt("model is wearing a garment",
                                                num_images_per_prompt=1, do_classifier_free_guidance=True,
                                                negative_prompt="low quality, blurry")
        pec, _, _, _ = pipe.encode_prompt("a photo of a garment", num_images_per_prompt=1,
                                          do_classifier_free_guidance=False)
        garm_t = s["cloth"].unsqueeze(0).to(device, dtype)
        out = pipe(prompt_embeds=pe, negative_prompt_embeds=npe, pooled_prompt_embeds=ppe,
                   negative_pooled_prompt_embeds=nppe, num_inference_steps=30, strength=1.0,
                   pose_img=pose_t, text_embeds_cloth=pec, cloth=garm_t,
                   mask_image=mask_pil, image=person_pil, height=cfg.height, width=cfg.width,
                   ip_adapter_image=cloth_pil, guidance_scale=2.0)[0][0]
        # side-by-side: garment | ground-truth person | generated
        combo = Image.new("RGB", (cfg.width*3, cfg.height))
        combo.paste(cloth_pil, (0,0)); combo.paste(person_pil, (cfg.width,0)); combo.paste(out, (cfg.width*2,0))
        panels.append(combo)
    if accelerator.trackers:
        accelerator.log({"val/samples": [wandb.Image(p) for p in panels]}, step=step)
    panels[0].save(os.path.join(cfg.output_dir, f"val_step{step}.png"))
    pipe.unet.train()
    print("validation rendered @ step", step)
# run once now to confirm it works (comment out if you only want it during training):
# run_validation(0)

In [ ]:
# 15. QUANTITATIVE evaluation on the test split: SSIM, LPIPS, (and FID over the set).
#     Run AFTER training. These are the numbers your professor wants in the results table.
import torch, numpy as np
from torchmetrics.image import StructuralSimilarityIndexMeasure
import lpips as lpips_lib

ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
lpips_metric = lpips_lib.LPIPS(net="alex").to(device)   # perceptual distance (lower=better)

@torch.no_grad()
def evaluate(n=50):
    pipe = _build_pipe(); pipe.unet.eval()
    test_ds = VTONDataset("test")
    ssim_vals, lpips_vals = [], []
    real_dir = os.path.join(cfg.output_dir, "fid_real"); fake_dir = os.path.join(cfg.output_dir, "fid_fake")
    os.makedirs(real_dir, exist_ok=True); os.makedirs(fake_dir, exist_ok=True)
    for i in range(min(n, len(test_ds))):
        s = test_ds[i]
        person_pil = Image.fromarray(((s["person"].permute(1,2,0).numpy()*0.5+0.5)*255).astype("uint8"))
        cloth_pil  = Image.fromarray(((s["cloth"].permute(1,2,0).numpy()*0.5+0.5)*255).astype("uint8"))
        mask_pil   = Image.fromarray((s["mask"][0].numpy()*255).astype("uint8"))
        pose_t = s["pose"].unsqueeze(0).to(device, dtype); garm_t = s["cloth"].unsqueeze(0).to(device, dtype)
        pe,npe,ppe,nppe = pipe.encode_prompt("model is wearing a garment", num_images_per_prompt=1,
                                             do_classifier_free_guidance=True, negative_prompt="low quality")
        pec,_,_,_ = pipe.encode_prompt("a photo of a garment", num_images_per_prompt=1, do_classifier_free_guidance=False)
        gen = pipe(prompt_embeds=pe, negative_prompt_embeds=npe, pooled_prompt_embeds=ppe,
                   negative_pooled_prompt_embeds=nppe, num_inference_steps=30, strength=1.0,
                   pose_img=pose_t, text_embeds_cloth=pec, cloth=garm_t, mask_image=mask_pil,
                   image=person_pil, height=cfg.height, width=cfg.width, ip_adapter_image=cloth_pil,
                   guidance_scale=2.0)[0][0]
        g = torch.tensor(np.array(gen)/255.).permute(2,0,1).unsqueeze(0).float().to(device)
        r = torch.tensor(np.array(person_pil)/255.).permute(2,0,1).unsqueeze(0).float().to(device)
        ssim_vals.append(ssim_metric(g, r).item())
        lpips_vals.append(lpips_metric(g*2-1, r*2-1).item())
        person_pil.save(os.path.join(real_dir, f"{i}.png")); gen.save(os.path.join(fake_dir, f"{i}.png"))
    print(f"SSIM  (higher better): {np.mean(ssim_vals):.4f}")
    print(f"LPIPS (lower  better): {np.mean(lpips_vals):.4f}")
    try:
        from cleanfid import fid
        print("FID   (lower  better):", fid.compute_fid(real_dir, fake_dir))
    except Exception as e:
        print("FID skipped (needs more samples / clean-fid):", e)
    if accelerator.trackers:
        accelerator.log({"eval/ssim": float(np.mean(ssim_vals)), "eval/lpips": float(np.mean(lpips_vals))})
# evaluate(50)

In [ ]:
# 16. Ship the result. The LoRA adapter is small; load it onto the base UNet to serve.
#     In the inference server (idm_vton_server.ipynb), after building TryonNet, add:
#
#        unet.load_adapter("/content/idmvton_lora_out/lora-final")    # our fine-tune
#
#     (Optionally merge for a standalone UNet: `unet = unet.merge_and_unload()`.)
print("LoRA adapter dir:", os.path.join(cfg.output_dir, "lora-final"))
!ls -la {cfg.output_dir}
# Copy to Drive so it survives the session:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r {cfg.output_dir} /content/drive/MyDrive/

## 17. Professor Q&A — questions you will likely be asked (with model answers)

**Q1. Why did you fine-tune with LoRA instead of full fine-tuning?**
Full fine-tuning of IDM-VTON updates two ~2.5B-param SDXL UNets; at 768×1024 with optimizer
state that exceeds ~40–60 GB VRAM and risks overfitting/catastrophic forgetting on a 10k set.
LoRA freezes the base weights and learns small rank-`r` updates on the attention projections
(<1% of params), so it fits on one GPU, trains faster, regularises better on small data, and
yields a portable ~10–100 MB adapter. Quality is close to full FT for domain adaptation.

**Q2. What is the loss function and where does it come from?**
The DDPM training objective: predict the Gaussian noise added to the latent. We add noise `ε`
to the clean latent `z₀` at timestep `t` (`z_t = √ᾱ_t z₀ + √(1−ᾱ_t) ε`) and minimise
`‖ε − ε̂_θ(z_t,t,cond)‖²`. This is the variational bound on the data likelihood reduced to a
simple reweighted MSE (Ho et al. 2020). We add **Min-SNR-γ** weighting (Hang et al. 2023) to
stop low-noise timesteps from dominating, which speeds convergence.

**Q3. Why predict noise (ε) and not the image directly?**
ε-prediction gives a well-conditioned, unit-variance target at every timestep, which trains
stably. (Alternatives: x₀-prediction and v-prediction; SDXL uses ε by default. If the loaded
scheduler is v-prediction, the target must be `scheduler.get_velocity(...)` — that's why the
code comments flag it.)

**Q4. What are the agnostic mask, DensePose, and OpenPose for?**
- **Agnostic mask**: the region to repaint (the old garment area). Built from **human parsing**
  (which pixels are torso/arms/legs) + **OpenPose** keypoints (to cut a clean, pose-aware region).
  It prevents the model from copying the original clothes and tells it where the new garment goes.
- **DensePose**: a dense map of body-surface coordinates; it anchors how the garment should
  wrap/drape over the 3D body, preserving pose and proportions.
These are *conditioning*, not learned — so they must be computed identically at train and test.

**Q5. How does the garment's appearance get into the result? (GarmentNet vs IP-Adapter)**
Two complementary paths: **GarmentNet** (a parallel UNet) encodes the flat garment and its
intermediate features are injected into TryonNet's self-attention — this carries *fine* texture,
patterns, logos. The **IP-Adapter** image encoder (CLIP-vision) injects a *global* garment
embedding via cross-attention — overall colour/style. Together they keep the garment faithful.

**Q6. Why are the VAE and text encoders frozen?**
They are general-purpose and already well-trained; fine-tuning them on 10k images would risk
degrading them and wastes memory. We only adapt the part that maps conditioning→garment-on-person
(TryonNet). The VAE just moves images ↔ latents; the text encoders just embed the prompt.

**Q7. What is SDXL "micro-conditioning" (added_cond_kwargs / time_ids)?**
SDXL conditions on the original/target size and crop offsets via `time_ids`, plus a pooled text
embedding (`text_embeds`). This lets it handle resolution/cropping consistently. We pass the
training resolution as the size ids.

**Q8. Train vs inference schedulers?**
We train with **DDPM** (1000 steps, the objective above). At inference we can use a faster
sampler (DDIM/Euler, ~30 steps) — same model, different way of integrating the reverse process.

**Q9. How do you prevent / detect overfitting? How is the data split?**
Hold out a **test** split (e.g. 1k pairs) never seen in training; watch validation SSIM/LPIPS
and the qualitative renders. LoRA + dropout + a modest step count + Min-SNR all regularise.
Signs of overfitting: train loss keeps dropping while val LPIPS rises / val renders memorise
training people.

**Q10. Mixed precision, gradient checkpointing, gradient accumulation, 8-bit Adam — why each?**
All are memory/throughput tricks to fit on one GPU: **bf16** halves activation memory (and is
numerically safer than fp16); **grad checkpointing** recomputes activations in the backward pass
instead of storing them; **grad accumulation** simulates a larger batch (effective `bs×accum`)
without the memory; **8-bit Adam** stores optimizer moments in int8.

**Q11. Which metrics did you report and what do they mean?**
- **SSIM** (↑): structural similarity to the ground-truth person — geometry/structure fidelity.
- **LPIPS** (↓): deep perceptual distance — matches human judgement of similarity better than SSIM.
- **FID** (↓): distribution distance between generated and real sets — overall realism.
Limitation: paired pixel metrics penalise *plausible* differences; that's why we also show
qualitative panels.

**Q12. How many steps/epochs and what learning rate?**
LoRA tolerates higher LR (1e-4) with cosine decay + warmup. With 9k pairs and effective batch 4,
one epoch ≈ 2250 steps; 6k steps ≈ ~2.7 epochs is a reasonable start — pick the checkpoint with
the best validation LPIPS rather than the last one.

**Q13. What were the main engineering challenges?**
Building correct conditioning (mask/DensePose) for our raw pairs, matching TryonNet's exact input
channel order (13ch) and the GarmentNet feature-injection interface, and fitting SDXL on one GPU
(hence LoRA + the memory tricks). Dependency pinning (numpy<2 for detectron2, diffusers 0.25.0)
was also critical.

**Q14. How would you improve results with more compute/data?**
Higher LoRA rank or full fine-tune on A100-80GB; also LoRA GarmentNet; more/cleaner paired data;
real per-garment captions; EMA of weights; classifier-free guidance training (randomly drop
conditioning); and higher-resolution training.


## 18. Run order & troubleshooting

**Order:** cell 1 → 2 → **Runtime: Restart session** → 3 → 4 → 5 → 6 (long, one-time) → 7 → 8 → 9 → 10 (W&B login) → 11 → 12 (training) → 13 → 14 → 15 → 16.

**Common issues**
- *`conv_in.in_channels` ≠ 13* (cell 8): your checkpoint packs conditioning differently — adjust the
  `torch.cat([...])` order/channels in cell 11 to match (the print tells you the total).
- *`unet(...) got unexpected keyword 'garment_features'`*: the kwarg name differs in this checkpoint.
  Print `inspect.signature(unet.forward)` (cell 8) and use the correct name (e.g. it may take the
  GarmentNet features as `down_block_additional_residuals` or be set via a forward hook). This is the
  one spot most likely to need a 1-line change for your exact weights.
- *CUDA OOM*: lower `width/height` to 512×384, `lora_rank` to 16, ensure `gradient_checkpointing=True`,
  keep `train_batch_size=1`, raise `grad_accum` instead.
- *DensePose `config does not exist`*: cwd must be `/content/IDM-VTON` and `./ckpt/densepose/...` must
  exist (symlinks from the inference notebook's model-load cell).
- *v-prediction scheduler*: set `target = noise_scheduler.get_velocity(latents, noise, t)` in cell 11.
